# Notebook 1: Data Exploration & Sensor Analysis
**Project:** Predictive Maintenance for O&G Drilling Rigs  
**Dataset:** NASA C-MAPSS Turbofan Degradation  
**Goal:** Understand sensor behavior and identify degradation patterns before failure

## 1. Setup

In [1]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11, 'axes.grid': True, 'grid.alpha': 0.3})
print("Libraries loaded.")

ModuleNotFoundError: No module named 'numpy'

## 2. Load Data

In [ ]:
from src.ingestion.data_loader import load_train_test, print_summary, OG_SENSOR_MAP, INFORMATIVE_SENSORS

train_df, test_df, true_rul = load_train_test('FD001')
print_summary(train_df, 'FD001')
train_df.head(3)

## 3. Lifecycle Distribution
How many operating cycles do units survive before failure?

In [ ]:
cycles_per_unit = train_df.groupby('unit_id')['cycle'].max()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(cycles_per_unit, bins=20, color='#378ADD', edgecolor='white', linewidth=0.5)
axes[0].set_xlabel('Lifecycle (cycles to failure)')
axes[0].set_ylabel('Number of units')
axes[0].set_title('Equipment Lifecycle Distribution (FD001)')
axes[0].axvline(cycles_per_unit.median(), color='#E24B4A', linestyle='--', label=f'Median: {cycles_per_unit.median():.0f}')
axes[0].legend()

axes[1].boxplot(cycles_per_unit, vert=False, patch_artist=True,
                boxprops=dict(facecolor='#B5D4F4'), medianprops=dict(color='#E24B4A'))
axes[1].set_xlabel('Lifecycle (cycles)')
axes[1].set_title('Lifecycle Boxplot')

plt.tight_layout()
plt.savefig('../docs/images/lifecycle_distribution.png', bbox_inches='tight')
plt.show()
print(f"Lifecycle stats:\n{cycles_per_unit.describe().round(1)}")

## 4. Sensor Degradation Patterns
Visualise how sensors change as equipment approaches failure. This is the core signal our model learns.

In [ ]:
KEY_SENSORS = ['sensor_2', 'sensor_4', 'sensor_7', 'sensor_11', 'sensor_20', 'sensor_21']

# Select 5 representative units
sample_units = train_df['unit_id'].unique()[:5]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for ax, sensor in zip(axes, KEY_SENSORS):
    for uid in sample_units:
        unit = train_df[train_df['unit_id'] == uid].sort_values('cycle')
        max_c = unit['cycle'].max()
        # Normalise x-axis to % of life
        ax.plot(unit['cycle'] / max_c * 100, unit[sensor],
                alpha=0.5, linewidth=0.9)
    ax.set_xlabel('% of lifecycle')
    ax.set_ylabel('Reading')
    og_name = OG_SENSOR_MAP.get(sensor, sensor)
    ax.set_title(f'{sensor}\n{og_name[:35]}', fontsize=9)

plt.suptitle('Sensor Trends Across Equipment Lifecycle (5 units)', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('../docs/images/sensor_degradation.png', bbox_inches='tight')
plt.show()

## 5. Correlation Analysis
Which sensors are most correlated with remaining useful life?

In [ ]:
from src.features.label_engineering import add_rul_labels

train_labeled = add_rul_labels(train_df.copy())

sensor_cols = [f'sensor_{i}' for i in range(1, 22)]
correlations = train_labeled[sensor_cols + ['rul']].corr()['rul'].drop('rul').sort_values()

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#E24B4A' if v < 0 else '#378ADD' for v in correlations]
bars = ax.barh(correlations.index, correlations.values, color=colors, edgecolor='none')
ax.axvline(0, color='#888', linewidth=0.8)
ax.set_xlabel('Pearson correlation with RUL')
ax.set_title('Sensor Correlation with Remaining Useful Life')

# Annotate informative sensors
for i, (sensor, val) in enumerate(correlations.items()):
    if sensor in INFORMATIVE_SENSORS:
        ax.get_yticklabels()[i].set_fontweight('bold')
        ax.get_yticklabels()[i].set_color('#0C447C')

plt.tight_layout()
plt.savefig('../docs/images/sensor_rul_correlation.png', bbox_inches='tight')
plt.show()
print("Bold blue sensors = selected as informative features")

## 6. Near-Failure vs Healthy: Signal Comparison
Compare sensor distributions in the last 20 cycles (near failure) vs early lifecycle (healthy).

In [ ]:
train_labeled = add_rul_labels(train_df.copy())
near_failure = train_labeled[train_labeled['rul'] <= 20]
healthy = train_labeled[train_labeled['rul'] >= 100]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for ax, sensor in zip(axes, KEY_SENSORS):
    ax.hist(healthy[sensor], bins=30, alpha=0.6, label='Healthy (RUL ≥ 100)',
            color='#1D9E75', density=True)
    ax.hist(near_failure[sensor], bins=30, alpha=0.6, label='Near failure (RUL ≤ 20)',
            color='#E24B4A', density=True)
    ax.set_title(f'{sensor}', fontsize=10)
    ax.set_ylabel('Density')
    ax.legend(fontsize=7)

plt.suptitle('Healthy vs Near-Failure Sensor Distributions', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('../docs/images/healthy_vs_failure.png', bbox_inches='tight')
plt.show()
print(f"Near-failure samples: {len(near_failure):,}")
print(f"Healthy samples:      {len(healthy):,}")

## 7. Correlation Heatmap (Informative Sensors)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
corr_matrix = train_df[INFORMATIVE_SENSORS].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, ax=ax, linewidths=0.3, annot_kws={'size': 8})
ax.set_title('Sensor Cross-Correlation Matrix')
plt.tight_layout()
plt.savefig('../docs/images/sensor_correlation_heatmap.png', bbox_inches='tight')
plt.show()

## 8. Key EDA Takeaways

- **sensor_4, sensor_11, sensor_12, sensor_20, sensor_21** show the clearest degradation trends correlated with RUL
- Distribution overlap between healthy and near-failure is significant — raw thresholds alone will produce ~38% false alarm rate (confirmed by baseline model)
- Lifecycle varies from ~120 to ~360 cycles, meaning time-series CV is essential
- **Next step:** Feature engineering → `02_feature_engineering.ipynb`